# Semantic Segmentation (PyTorch Version)

![](https://miro.medium.com/max/700/1*8Nwk_IdGpe235Nsfewpucg.png)

from Fei-Fei Li Stanford Course — Detection And Segmentation

# STEP 1: 데이터셋 읽어들이기

https://www.kaggle.com/nikhilpandey360/lung-segmentation-from-chest-x-ray-dataset

![](https://www.altoros.com/blog/wp-content/uploads/2018/12/segmentation-results-max-dice-score.png)

### Lung_Segmentation.zip
### 256x256x3
### 566 [image, label]

In [ ]:
!rm -rf *
!wget https://github.com/mi2rl/datasets/raw/master/Lung_Segmentation.zip
!unzip Lung_Segmentation.zip

In [ ]:
import numpy as np
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from skimage.io import imread
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

IMG_WIDTH = 256
IMG_HEIGHT = 256
IMG_CHANNELS = 3

data_path = './Lung_Segmentation'

files = os.listdir(os.path.join(data_path, 'image'))
file_headers = [os.path.splitext(f)[0] for f in files]

X_all = np.zeros((len(file_headers), IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), dtype=np.uint8)
y_all = np.zeros((len(file_headers), IMG_HEIGHT, IMG_WIDTH, 1), dtype=bool)

for count, fh in enumerate(file_headers):
    f1 = os.path.join(data_path, 'image', '{}.png'.format(fh))
    l1 = os.path.join(data_path, 'label', '{}.png'.format(fh))
    img = imread(f1)[:, :, :IMG_CHANNELS]
    mask = imread(l1)
    mask = np.expand_dims(mask, axis=-1)
    X_all[count] = img
    y_all[count] = mask

print('Loaded:', X_all.shape, y_all.shape)

**딥러닝을 위한 데이터 전처리**

In [ ]:
X_all = X_all.astype('float32') / 255.

**학습, 검증, 테스트 데이터 셋으로 분리**

In [ ]:
seed = 7
np.random.seed(seed)

X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=seed)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=seed)

print('X_train', X_train.shape)
print('X_valid', X_valid.shape)
print('X_test ', X_test.shape)
print('y_train', y_train.shape)
print('y_valid', y_valid.shape)
print('y_test ', y_test.shape)

**PyTorch Dataset & DataLoader**

PyTorch는 (N, H, W, C) 대신 **채널 우선(channel-first)** 형식 **(N, C, H, W)** 을 사용합니다.

In [ ]:
class LungDataset(Dataset):
    """X: (N,H,W,C) float32, y: (N,H,W,1) bool → PyTorch tensors (N,C,H,W)"""
    def __init__(self, X, y):
        # permute: (N,H,W,C) -> (N,C,H,W)
        self.X = torch.FloatTensor(X).permute(0, 3, 1, 2)
        self.y = torch.FloatTensor(y.astype(np.float32)).permute(0, 3, 1, 2)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


BATCH_SIZE = 8
train_loader = DataLoader(LungDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(LungDataset(X_valid, y_valid), batch_size=BATCH_SIZE)
test_loader  = DataLoader(LungDataset(X_test,  y_test),  batch_size=BATCH_SIZE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# STEP 2: 데이터 살펴보기

In [ ]:
def plotTrainData(X, y, split_name):
    for _ in range(3):
        ix = np.random.randint(0, len(X))
        plt.subplot(1, 2, 1)
        plt.title('X_' + split_name)
        plt.imshow(X[ix])
        plt.axis('off')
        plt.subplot(1, 2, 2)
        plt.title('y_' + split_name)
        plt.imshow(np.squeeze(y[ix]), 'gray')
        plt.axis('off')
        plt.show()

plotTrainData(X_train, y_train, 'train')
plotTrainData(X_valid, y_valid, 'valid')
plotTrainData(X_test,  y_test,  'test')

# STEP 3: VGG16 네트워크 다시보기

![VGG16](https://neurohive.io/wp-content/uploads/2018/11/vgg16-1-e1542731207177.png)

PyTorch에서는 `nn.Module`을 상속받아 모델을 정의합니다.  
Keras의 `Model(inputs, outputs)` 방식과 달리, `forward()` 메서드에서 연산 흐름을 직접 작성합니다.

In [ ]:
def cbr(in_c, out_c):
    """Conv2d -> BatchNorm2d -> ReLU 블록"""
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True)
    )


class VGG16(nn.Module):
    def __init__(self, num_classes=1000):
        super().__init__()
        # Block 1: 256x256 -> 128x128
        self.block1 = nn.Sequential(cbr(3, 64), cbr(64, 64))
        self.pool1  = nn.MaxPool2d(2)
        # Block 2: 128x128 -> 64x64
        self.block2 = nn.Sequential(cbr(64, 128), cbr(128, 128))
        self.pool2  = nn.MaxPool2d(2)
        # Block 3: 64x64 -> 32x32
        self.block3 = nn.Sequential(cbr(128, 256), cbr(256, 256), cbr(256, 256))
        self.pool3  = nn.MaxPool2d(2)
        # Block 4: 32x32 -> 16x16
        self.block4 = nn.Sequential(cbr(256, 512), cbr(512, 512), cbr(512, 512))
        self.pool4  = nn.MaxPool2d(2)
        # Block 5: 16x16 -> 8x8
        self.block5 = nn.Sequential(cbr(512, 512), cbr(512, 512), cbr(512, 512))
        self.pool5  = nn.MaxPool2d(2)
        # Fully-connected 부분 (분류용)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((7, 7)),
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, 4096), nn.ReLU(inplace=True),
            nn.Linear(4096, 4096),        nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        x = self.pool1(self.block1(x))
        x = self.pool2(self.block2(x))
        x = self.pool3(self.block3(x))
        x = self.pool4(self.block4(x))
        x = self.pool5(self.block5(x))
        return self.classifier(x)


vgg = VGG16()
print(vgg)

# 공통 유틸리티: Dice 손실, 학습 루프, 시각화

**Dice Coefficient**

![Dice](https://miro.medium.com/max/858/1*yUd5ckecHjWZf6hGrdlwzA.png)

In [ ]:
def dice_coef(y_true, y_pred):
    y_true_f = y_true.float().view(-1)
    y_pred_f = y_pred.view(-1)
    intersection = (y_true_f * y_pred_f).sum()
    return (2.0 * intersection + 1.0) / (y_true_f.sum() + y_pred_f.sum() + 1.0)

def dice_coef_loss(y_true, y_pred):
    return -dice_coef(y_true, y_pred)

def dice_coef_numpy(y_true, y_pred):
    """numpy 배열 기반 Dice (시각화용)"""
    y_t = y_true.flatten().astype(np.float32)
    y_p = y_pred.flatten().astype(np.float32)
    return (2.0 * np.sum(y_t * y_p) + 1.0) / (np.sum(y_t) + np.sum(y_p) + 1.0)

def pixel_accuracy(y_true, y_pred):
    pred_bin = (y_pred > 0.5).float()
    return (pred_bin == y_true).float().mean().item()


def train_model(model, train_loader, val_loader, epochs=20, lr=0.01):
    model = model.to(device)
    optimizer = torch.optim.SGD(
        model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-6, nesterov=True
    )
    history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}

    for epoch in range(epochs):
        model.train()
        tr_loss, tr_acc = 0.0, 0.0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = model(Xb)
            loss = dice_coef_loss(yb, pred)
            loss.backward()
            optimizer.step()
            tr_loss += loss.item()
            tr_acc  += pixel_accuracy(yb, pred)

        model.eval()
        vl_loss, vl_acc = 0.0, 0.0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                pred = model(Xb)
                vl_loss += dice_coef_loss(yb, pred).item()
                vl_acc  += pixel_accuracy(yb, pred)

        nt, nv = len(train_loader), len(val_loader)
        history['loss'].append(tr_loss / nt)
        history['val_loss'].append(vl_loss / nv)
        history['accuracy'].append(tr_acc / nt)
        history['val_accuracy'].append(vl_acc / nv)
        print(f'Epoch {epoch+1:02d}/{epochs}  '
              f'loss={history["loss"][-1]:.4f}  acc={history["accuracy"][-1]:.4f}  '
              f'val_loss={history["val_loss"][-1]:.4f}  val_acc={history["val_accuracy"][-1]:.4f}')

    return history


def plot_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history['loss'],     'y', label='train loss')
    ax1.plot(history['val_loss'], 'r', label='val loss')
    ax1.set_xlabel('epoch'); ax1.set_ylabel('loss'); ax1.legend()
    ax2.plot(history['accuracy'],     'b', label='train acc')
    ax2.plot(history['val_accuracy'], 'g', label='val acc')
    ax2.set_xlabel('epoch'); ax2.set_ylabel('accuracy'); ax2.legend()
    plt.show()


def plotPredictions(X_tr, y_tr, X_vl, y_vl, X_te, y_te, model):
    model.eval()
    for X_, y_, name in [(X_tr, y_tr, 'train'), (X_vl, y_vl, 'valid'), (X_te, y_te, 'test')]:
        ix   = np.random.randint(0, len(X_))
        img  = X_[ix]                      # (H,W,C)
        mask = y_[ix, :, :, 0]             # (H,W)
        inp  = torch.FloatTensor(img).permute(2, 0, 1).unsqueeze(0).to(device)
        with torch.no_grad():
            pred_t = model(inp)
        pred_np  = pred_t.cpu().numpy()[0, 0]       # (H,W)
        pred_bin = (pred_np > 0.5).astype(np.uint8)
        dice     = dice_coef_numpy(mask.astype(np.float32), pred_np)

        plt.figure(figsize=(12, 4))
        plt.subplot(1, 3, 1); plt.title(f'X_{name}');   plt.axis('off'); plt.imshow(img)
        plt.subplot(1, 3, 2); plt.title(f'Y_{name}');   plt.axis('off'); plt.imshow(mask, 'gray')
        plt.subplot(1, 3, 3); plt.title(f'Pred Dice={dice:.4f}'); plt.axis('off'); plt.imshow(pred_bin, 'gray')
        plt.show()

# STEP 4: 첫번째 영상분할 모델 (FCN32s)

VGG 인코더 + 32배 업샘플링으로 바로 원본 해상도 복원

## 업샘플링(Upsampling)과 이미지 보간(Image Interpolation)

저해상도 feature map(8×8)을 원본 크기(256×256)로 복원할 때 **보간(interpolation)** 방법이 사용됩니다.

![image interpolation](https://matplotlib.org/_images/interpolation_methods.png)

| 보간 방법 | 설명 | 특징 |
|-----------|------|------|
| **Nearest** | 가장 가까운 픽셀 값을 그대로 복사 | 빠르지만 블록 아티팩트(격자 무늬) 발생 |
| **Bilinear** | 주변 4개 픽셀의 거리 가중 평균 | 부드럽고 속도·품질 균형이 좋아 FCN에서 사용 |
| **Bicubic** | 주변 16개 픽셀의 3차 함수 가중 평균 | 가장 부드럽지만 느림 |

PyTorch에서는 `nn.Upsample(scale_factor=32, mode='bilinear', align_corners=False)` 로 설정합니다.

> **핵심 — 채널 축소 후 업샘플링**  
> 512채널을 업샘플 하면 `512 × (256×256)` 크기의 텐서를 처리해야 합니다.  
> **먼저 1×1 conv로 채널을 1로 줄인 뒤** 업샘플하면 연산량이 **512배** 감소하며,  
> 이는 이후 FCN8s·FCN2s의 score 레이어 설계와도 동일한 방식입니다.

In [ ]:
class FCN32s(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(cbr(3, 64),   cbr(64, 64));  self.pool1 = nn.MaxPool2d(2)
        self.block2 = nn.Sequential(cbr(64, 128), cbr(128, 128)); self.pool2 = nn.MaxPool2d(2)
        self.block3 = nn.Sequential(cbr(128, 256), cbr(256, 256), cbr(256, 256)); self.pool3 = nn.MaxPool2d(2)
        self.block4 = nn.Sequential(cbr(256, 512), cbr(512, 512), cbr(512, 512)); self.pool4 = nn.MaxPool2d(2)
        self.block5 = nn.Sequential(cbr(512, 512), cbr(512, 512), cbr(512, 512)); self.pool5 = nn.MaxPool2d(2)
        # 채널을 먼저 1로 줄인 뒤 x32 업샘플링 (FCN8s/FCN2s와 동일한 방식)
        self.score = nn.Conv2d(512, 1, kernel_size=1)
        self.up32  = nn.Upsample(scale_factor=32, mode='bilinear', align_corners=False)

    def forward(self, x):
        x = self.pool1(self.block1(x))   # 128x128
        x = self.pool2(self.block2(x))   # 64x64
        x = self.pool3(self.block3(x))   # 32x32
        x = self.pool4(self.block4(x))   # 16x16
        x = self.pool5(self.block5(x))   # 8x8, 512ch
        x = self.score(x)                # 8x8, 1ch  ← 채널 축소 먼저
        x = self.up32(x)                 # 256x256, 1ch
        return torch.sigmoid(x)

In [ ]:
model_fcn32s = FCN32s()
hist_fcn32s  = train_model(model_fcn32s, train_loader, valid_loader, epochs=20)
torch.save(model_fcn32s.state_dict(), 'fcn32s.pth')
plot_history(hist_fcn32s)

# STEP 5: 결과 확인하기

In [ ]:
plotPredictions(X_train, y_train, X_valid, y_valid, X_test, y_test, model_fcn32s)

# STEP 6: 두번째 모델 (FCN8s) — Skip Connection

![FCN schema](https://raw.githubusercontent.com/YimianDai/images/master/fcn_schema.png)

pool3, pool4 feature map을 디코더에 연결하여 세밀한 공간 정보 복원

In [ ]:
class FCN8s(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(cbr(3, 64),   cbr(64, 64));   self.pool1 = nn.MaxPool2d(2)
        self.block2 = nn.Sequential(cbr(64, 128), cbr(128, 128)); self.pool2 = nn.MaxPool2d(2)
        self.block3 = nn.Sequential(cbr(128, 256), cbr(256, 256), cbr(256, 256)); self.pool3 = nn.MaxPool2d(2)
        self.block4 = nn.Sequential(cbr(256, 512), cbr(512, 512), cbr(512, 512)); self.pool4 = nn.MaxPool2d(2)
        self.block5 = nn.Sequential(cbr(512, 512), cbr(512, 512), cbr(512, 512)); self.pool5 = nn.MaxPool2d(2)

        # Bottleneck 1x1 convolutions
        self.conv6 = nn.Sequential(nn.Conv2d(512, 2048, 1), nn.ReLU(inplace=True))
        self.conv7 = nn.Sequential(nn.Conv2d(2048, 2048, 1), nn.ReLU(inplace=True))
        self.conv8 = nn.Sequential(nn.Conv2d(2048, 1, 1),   nn.ReLU(inplace=True))

        # Skip projection
        self.score4 = nn.Sequential(nn.Conv2d(512, 1, 1), nn.ReLU(inplace=True))
        self.score3 = nn.Sequential(nn.Conv2d(256, 1, 1), nn.ReLU(inplace=True))

        self.up2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.up8 = nn.Upsample(scale_factor=8, mode='bilinear', align_corners=False)

    def forward(self, x):
        x  = self.pool1(self.block1(x))    # 128x128
        x  = self.pool2(self.block2(x))    # 64x64
        p3 = self.pool3(self.block3(x))    # 32x32, 256ch
        p4 = self.pool4(self.block4(p3))   # 16x16, 512ch
        p5 = self.pool5(self.block5(p4))   # 8x8,  512ch

        x = self.conv8(self.conv7(self.conv6(p5)))   # 8x8, 1ch
        x = self.up2(x) + self.score4(p4)           # 16x16
        x = self.up2(x) + self.score3(p3)           # 32x32
        x = self.up8(x)                             # 256x256
        return torch.sigmoid(x)

In [ ]:
model_fcn8s = FCN8s()
hist_fcn8s  = train_model(model_fcn8s, train_loader, valid_loader, epochs=20)
torch.save(model_fcn8s.state_dict(), 'fcn8s.pth')
plot_history(hist_fcn8s)

In [ ]:
plotPredictions(X_train, y_train, X_valid, y_valid, X_test, y_test, model_fcn8s)

# STEP 6-1: 두번째 모델의 개선 (FCN2s) — 더 많은 Skip Connection

pool1~pool4까지 모두 skip connection으로 연결하여 고해상도 세부 정보 복원

In [ ]:
class FCN2s(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(cbr(3, 64),   cbr(64, 64));   self.pool1 = nn.MaxPool2d(2)
        self.block2 = nn.Sequential(cbr(64, 128), cbr(128, 128)); self.pool2 = nn.MaxPool2d(2)
        self.block3 = nn.Sequential(cbr(128, 256), cbr(256, 256), cbr(256, 256)); self.pool3 = nn.MaxPool2d(2)
        self.block4 = nn.Sequential(cbr(256, 512), cbr(512, 512), cbr(512, 512)); self.pool4 = nn.MaxPool2d(2)
        self.block5 = nn.Sequential(cbr(512, 512), cbr(512, 512), cbr(512, 512)); self.pool5 = nn.MaxPool2d(2)

        self.conv6 = nn.Sequential(nn.Conv2d(512, 2048, 1), nn.ReLU(inplace=True))
        self.conv7 = nn.Sequential(nn.Conv2d(2048, 2048, 1), nn.ReLU(inplace=True))
        self.conv8 = nn.Sequential(nn.Conv2d(2048, 1, 1),   nn.ReLU(inplace=True))

        self.score4 = nn.Sequential(nn.Conv2d(512, 1, 1), nn.ReLU(inplace=True))
        self.score3 = nn.Sequential(nn.Conv2d(256, 1, 1), nn.ReLU(inplace=True))
        self.score2 = nn.Sequential(nn.Conv2d(128, 1, 1), nn.ReLU(inplace=True))
        self.score1 = nn.Sequential(nn.Conv2d(64,  1, 1), nn.ReLU(inplace=True))

        self.up2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)

    def forward(self, x):
        p1 = self.pool1(self.block1(x))    # 128x128, 64ch
        p2 = self.pool2(self.block2(p1))   # 64x64,  128ch
        p3 = self.pool3(self.block3(p2))   # 32x32,  256ch
        p4 = self.pool4(self.block4(p3))   # 16x16,  512ch
        p5 = self.pool5(self.block5(p4))   # 8x8,    512ch

        x = self.conv8(self.conv7(self.conv6(p5)))   # 8x8
        x = self.up2(x) + self.score4(p4)           # 16x16
        x = self.up2(x) + self.score3(p3)           # 32x32
        x = self.up2(x) + self.score2(p2)           # 64x64
        x = self.up2(x) + self.score1(p1)           # 128x128
        x = self.up2(x)                             # 256x256
        return torch.sigmoid(x)

In [ ]:
model_fcn2s = FCN2s()
hist_fcn2s  = train_model(model_fcn2s, train_loader, valid_loader, epochs=20)
torch.save(model_fcn2s.state_dict(), 'fcn2s.pth')
plot_history(hist_fcn2s)

In [ ]:
plotPredictions(X_train, y_train, X_valid, y_valid, X_test, y_test, model_fcn2s)

# STEP 7: 세번째 모델 (FCN8s with Deconvolution)

Bilinear upsample 대신 **ConvTranspose2d** (학습 가능한 역합성곱)을 사용

![Deconvolution](https://miro.medium.com/max/1086/1*AbCrAqPBfkqGRdhKtiZQqA.png)

In [ ]:
def deconv_block(in_c, stride):
    """ConvTranspose2d + 2x Conv-BN-ReLU 블록"""
    return nn.Sequential(
        nn.ConvTranspose2d(in_c, 1, kernel_size=stride, stride=stride, padding=0),
        nn.Conv2d(1, 1, 3, padding=1), nn.BatchNorm2d(1), nn.ReLU(inplace=True),
        nn.Conv2d(1, 1, 3, padding=1), nn.BatchNorm2d(1), nn.ReLU(inplace=True),
    )


class FCN8sDeconv(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(cbr(3, 64),   cbr(64, 64));   self.pool1 = nn.MaxPool2d(2)
        self.block2 = nn.Sequential(cbr(64, 128), cbr(128, 128)); self.pool2 = nn.MaxPool2d(2)
        self.block3 = nn.Sequential(cbr(128, 256), cbr(256, 256), cbr(256, 256)); self.pool3 = nn.MaxPool2d(2)
        self.block4 = nn.Sequential(cbr(256, 512), cbr(512, 512), cbr(512, 512)); self.pool4 = nn.MaxPool2d(2)
        self.block5 = nn.Sequential(cbr(512, 512), cbr(512, 512), cbr(512, 512)); self.pool5 = nn.MaxPool2d(2)

        self.conv6 = nn.Sequential(nn.Conv2d(512, 2048, 1), nn.ReLU(inplace=True))
        self.conv7 = nn.Sequential(nn.Conv2d(2048, 2048, 1), nn.ReLU(inplace=True))
        self.conv8 = nn.Sequential(nn.Conv2d(2048, 1, 1),   nn.ReLU(inplace=True))

        self.score4 = nn.Sequential(nn.Conv2d(512, 1, 1), nn.ReLU(inplace=True))
        self.score3 = nn.Sequential(nn.Conv2d(256, 1, 1), nn.ReLU(inplace=True))

        # 학습 가능한 역합성곱 (deconvolution)
        self.deconv1 = deconv_block(1, stride=2)   # 8x8  -> 16x16
        self.deconv2 = deconv_block(1, stride=2)   # 16x16 -> 32x32
        self.deconv3 = deconv_block(1, stride=8)   # 32x32 -> 256x256

    def forward(self, x):
        x  = self.pool1(self.block1(x))
        x  = self.pool2(self.block2(x))
        p3 = self.pool3(self.block3(x))
        p4 = self.pool4(self.block4(p3))
        p5 = self.pool5(self.block5(p4))

        x = self.conv8(self.conv7(self.conv6(p5)))
        x = self.deconv1(x) + self.score4(p4)    # 16x16
        x = self.deconv2(x) + self.score3(p3)    # 32x32
        x = self.deconv3(x)                      # 256x256
        return torch.sigmoid(x)

In [ ]:
model_fcn8s_deconv = FCN8sDeconv()
hist_fcn8s_deconv  = train_model(model_fcn8s_deconv, train_loader, valid_loader, epochs=20)
torch.save(model_fcn8s_deconv.state_dict(), 'fcn8s_deconv.pth')
plot_history(hist_fcn8s_deconv)

In [ ]:
plotPredictions(X_train, y_train, X_valid, y_valid, X_test, y_test, model_fcn8s_deconv)

# STEP 8: 마지막 모델 (U-Net) — Concatenation Skip Connection

![UNet](https://www.renom.jp/notebooks/tutorial/image_processing/u-net/unet.png)

FCN의 **Add** 방식과 달리 U-Net은 인코더 feature map을 디코더에 **Concatenate** 하여 더 풍부한 정보 전달

## Skip Connection 비교: FCN(Add) vs U-Net(Concatenate)

![FCN skip connection 비교](http://2rct3i2488gxf9jvb1lqhek9-wpengine.netdna-ssl.com/wp-content/uploads/2017/05/fcn.png)

FCN과 U-Net의 핵심 차이는 인코더 feature map을 디코더에 연결하는 방식에 있습니다.

| 방식 | FCN (Add) | U-Net (Concatenate) |
|------|-----------|---------------------|
| 연산 | 업샘플 feature **+** 인코더 feature | `[업샘플 feature, 인코더 feature]` 채널 방향 병합 |
| 채널 수 | 변화 없음 | 2배로 증가 → 이후 Conv로 원하는 수로 축소 |
| 정보 보존 | element-wise 합산 (정보 일부 손실 가능) | 두 feature를 **완전히 보존**하여 전달 |
| 메모리 | 적음 | FCN보다 많음 |

```python
# FCN: element-wise add → 채널 수 유지
out = upsample(x) + score(skip_feature)

# U-Net: channel-wise concatenate → 채널 2배 → conv로 축소
out = conv_block(torch.cat([upsample(x), skip_feature], dim=1))
```

U-Net의 인코더–디코더 대칭 구조 덕분에 **고해상도 경계 정보**(인코더)와 **의미론적 위치 정보**(디코더)가 함께 활용되어, 의료 영상처럼 정밀한 경계가 중요한 분할 task에서 특히 강점을 보입니다.

![UNet architecture](https://www.renom.jp/notebooks/tutorial/image_processing/u-net/unet.png)

In [ ]:
def cbr2(in_c, out_c):
    """2-layer Conv-BN-ReLU 블록"""
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder
        self.enc1 = cbr2(3,   32);  self.pool1 = nn.MaxPool2d(2)
        self.enc2 = cbr2(32,  64);  self.pool2 = nn.MaxPool2d(2)
        self.enc3 = cbr2(64,  128); self.pool3 = nn.MaxPool2d(2)
        self.enc4 = cbr2(128, 256); self.pool4 = nn.MaxPool2d(2)
        self.bridge = cbr2(256, 512)
        # Decoder (ConvTranspose2d + Concatenate + cbr2)
        self.up4  = nn.ConvTranspose2d(512, 256, 2, stride=2); self.dec4 = cbr2(512, 256)
        self.up3  = nn.ConvTranspose2d(256, 128, 2, stride=2); self.dec3 = cbr2(256, 128)
        self.up2  = nn.ConvTranspose2d(128,  64, 2, stride=2); self.dec2 = cbr2(128,  64)
        self.up1  = nn.ConvTranspose2d( 64,  32, 2, stride=2); self.dec1 = cbr2( 64,  32)
        self.out  = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)                   # 256x256, 32ch
        e2 = self.enc2(self.pool1(e1))      # 128x128, 64ch
        e3 = self.enc3(self.pool2(e2))      #  64x64, 128ch
        e4 = self.enc4(self.pool3(e3))      #  32x32, 256ch
        b  = self.bridge(self.pool4(e4))    #  16x16, 512ch

        d4 = self.dec4(torch.cat([self.up4(b),  e4], dim=1))  # 32x32
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))  # 64x64
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))  # 128x128
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))  # 256x256
        return torch.sigmoid(self.out(d1))

In [ ]:
model_unet = UNet()
hist_unet  = train_model(model_unet, train_loader, valid_loader, epochs=20)
torch.save(model_unet.state_dict(), 'unet.pth')
plot_history(hist_unet)

In [ ]:
plotPredictions(X_train, y_train, X_valid, y_valid, X_test, y_test, model_unet)

# STEP 9: ResNet50-UNet (전이학습)

ImageNet 사전학습된 **ResNet50** 인코더 + U-Net 스타일 디코더

- Keras: `ResNet50(include_top=False, weights='imagenet')`
- PyTorch: `torchvision.models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)`

| Keras 레이어명          | PyTorch 레이어           | 출력 크기        |
|-------------------------|--------------------------|------------------|
| input_2                 | x (입력)                 | 256×256, 3ch     |
| conv1_relu              | conv1+bn1+relu           | 128×128, 64ch    |
| conv2_block3_out        | layer1                   | 64×64, 256ch     |
| conv3_block4_out        | layer2                   | 32×32, 512ch     |
| conv4_block6_out        | layer3                   | 16×16, 1024ch    |
| conv5_block3_out        | layer4 (bridge)          | 8×8, 2048ch      |

In [ ]:
class ResDecoderBlock(nn.Module):
    """ConvTranspose2d로 2배 업샘플 → skip feature concat → conv_block"""
    def __init__(self, in_c, skip_c, out_c):
        super().__init__()
        self.upconv = nn.ConvTranspose2d(in_c, out_c, kernel_size=2, stride=2)
        self.conv   = nn.Sequential(
            nn.Conv2d(out_c + skip_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),          nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        return self.conv(torch.cat([self.upconv(x), skip], dim=1))


class ResNet50UNet(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

        # Encoder — ResNet50 레이어 재사용
        self.enc_s2 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)  # 128x128, 64ch
        self.enc_pool = resnet.maxpool                                        # 64x64
        self.enc_s3   = resnet.layer1   # 64x64, 256ch
        self.enc_s4   = resnet.layer2   # 32x32, 512ch
        self.enc_s5   = resnet.layer3   # 16x16, 1024ch
        self.enc_b1   = resnet.layer4   # 8x8,  2048ch  (bridge)

        # Decoder — ResDecoderBlock(in_c, skip_c, out_c)
        self.d1 = ResDecoderBlock(2048, 1024, 1024)  # 16x16
        self.d2 = ResDecoderBlock(1024,  512,  512)  # 32x32
        self.d3 = ResDecoderBlock( 512,  256,  256)  # 64x64
        self.d4 = ResDecoderBlock( 256,   64,  128)  # 128x128
        self.d5 = ResDecoderBlock( 128,    3,   64)  # 256x256 (skip=원본 입력 3ch)

        self.out = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):
        s1 = x                                      # 256x256, 3ch
        s2 = self.enc_s2(x)                         # 128x128, 64ch
        s3 = self.enc_s3(self.enc_pool(s2))         # 64x64,  256ch
        s4 = self.enc_s4(s3)                        # 32x32,  512ch
        s5 = self.enc_s5(s4)                        # 16x16, 1024ch
        b1 = self.enc_b1(s5)                        # 8x8,  2048ch

        d1 = self.d1(b1, s5)                        # 16x16, 1024ch
        d2 = self.d2(d1, s4)                        # 32x32,  512ch
        d3 = self.d3(d2, s3)                        # 64x64,  256ch
        d4 = self.d4(d3, s2)                        # 128x128, 128ch
        d5 = self.d5(d4, s1)                        # 256x256,  64ch

        return torch.sigmoid(self.out(d5))

In [ ]:
model_res_unet = ResNet50UNet()
hist_res_unet  = train_model(model_res_unet, train_loader, valid_loader, epochs=40)
torch.save(model_res_unet.state_dict(), 'res_unet.pth')
plot_history(hist_res_unet)

In [ ]:
plotPredictions(X_train, y_train, X_valid, y_valid, X_test, y_test, model_res_unet)

## 모델별 비교 요약

| 모델              | 디코더 방식          | Skip Connection |
|-------------------|---------------------|-----------------|
| FCN32s            | Bilinear x32        | 없음            |
| FCN8s             | Bilinear x2, x8     | pool3, pool4 (Add) |
| FCN2s             | Bilinear x2 x5회    | pool1~pool4 (Add) |
| FCN8s-Deconv      | ConvTranspose2d     | pool3, pool4 (Add) |
| U-Net             | ConvTranspose2d     | enc1~enc4 (Concatenate) |
| ResNet50-UNet     | ConvTranspose2d     | ResNet50 레이어 (Concatenate) + 전이학습 |